In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

### PPI

In [ ]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [ ]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

In [ ]:
protein_interaction

### DrugBank

In [ ]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [ ]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        
        # Extract toxicity information at drug level
        toxicity = drug.find('db:toxicity', ns)
        toxicity_text = toxicity.text if toxicity is not None else None
        
        # Extract categories (may contain toxicity classifications)
        categories = drug.findall('db:categories/db:category', ns)
        category_list = []
        for cat in categories:
            cat_name = cat.find('db:category', ns)
            if cat_name is not None:
                category_list.append(cat_name.text)
        categories_str = ', '.join(category_list) if category_list else None
        
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None,
                    'toxicity': toxicity_text,
                    'categories': categories_str
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None,
                        'toxicity': toxicity_text,
                        'categories': categories_str
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [ ]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')
Drug_bank['gene'] = Drug_bank['gene'].str.upper()  # Convert gene names to uppercase for consistency

In [ ]:
### drugbank statistics
print(f"Total unique drugs: {Drug_bank['drug'].nunique()}")
print(f"Total unique genes: {Drug_bank['gene'].nunique()}")
print(f"Total unique drug-gene interactions: {len(Drug_bank)}")

In [ ]:
Drug_bank

### Genetic Results

In [ ]:
### import data
### genes
hpv_positive_amplification_top_gene_df= pd.read_csv('Results/CNV results/HPV positive amplification top genes.csv')
hpv_positive_amplification_top_gene_df['MUT_TYPE'] = 'AMPLIFICATION'
hpv_positive_deletion_top_gene_df= pd.read_csv('Results/CNV results/HPV positive deletion top genes.csv')
hpv_positive_deletion_top_gene_df['MUT_TYPE'] = 'DELETION'
hpv_positive_deletion_top_gene_df['amplification_count'] = hpv_positive_deletion_top_gene_df['deletion_count']
hpv_positive_genes = pd.concat([hpv_positive_amplification_top_gene_df, hpv_positive_deletion_top_gene_df], axis=0)
hpv_positive_genes =hpv_positive_genes[['gene_name', 'gene', 'Sample_size', 'amplification_count',
       'frequency_percentage', 'gistic_score', 'p_value', 'q_value',
       'significant', 'empirical_p_value', 'empirical_q_value',
       'empirical_significant', 'MUT_TYPE']]

hpv_negative_amplification_top_gene_df = pd.read_csv('Results/CNV results/HPV negative amplification top genes.csv')
hpv_negative_amplification_top_gene_df['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_deletion_top_gene_df = pd.read_csv('Results/CNV results/HPV negative deletion top genes.csv')
hpv_negative_deletion_top_gene_df['MUT_TYPE'] = 'DELETION'
hpv_negative_deletion_top_gene_df['amplification_count'] = hpv_negative_deletion_top_gene_df['deletion_count']
hpv_negative_genes = pd.concat([hpv_negative_amplification_top_gene_df, hpv_negative_deletion_top_gene_df], axis=0)
hpv_negative_genes =hpv_negative_genes[['gene_name', 'gene', 'Sample_size', 'amplification_count',
       'frequency_percentage', 'gistic_score', 'p_value', 'q_value',
       'significant', 'empirical_p_value', 'empirical_q_value',
       'empirical_significant', 'MUT_TYPE']]

hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')

### EHR results

In [ ]:
Drug_bank[Drug_bank['drug']=="Polyethylene glycol"]

In [ ]:
ehr_hpv_pos_drugs = pd.read_csv('../1. EHR based drug repurposing/Results/ML analysis/hpv_positive_ml_drug_xgb_results.csv')
ehr_hpv_neg_drugs = pd.read_csv('../1. EHR based drug repurposing/Results/ML analysis/hpv_negative_ml_drug_xgb_results.csv')

### replace "_aspirin" in ehr_hpv_neg_drugs with "_acetylsalicylic acid"
ehr_hpv_neg_drugs['feature'] = ehr_hpv_neg_drugs['feature'].str.replace('_aspirin', '_acetylsalicylic acid')
ehr_hpv_pos_drugs['feature'] = ehr_hpv_pos_drugs['feature'].str.replace('_sennosides, USP', '_sennosides')

### Functions

In [ ]:
def connect_to_genes(gene_df, ppi = protein_interaction):
    ### connect genes to those in PPI
    ### ensure only strong interactions are considered
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()
    ppi['Translated_protein_1'] = ppi['Translated_protein_1'].str.upper()
    ppi['Translated_protein_2'] = ppi['Translated_protein_2'].str.upper()
    
    ppi = ppi[ppi['combined_score']>=700].reset_index(drop=True)
    ppi_available_genes = set(ppi['Translated_protein_1']).union(set(ppi['Translated_protein_2']))
    #### get literature validated genes
    for i, row in tqdm(gene_df.iterrows(), total=gene_df.shape[0]):
        connected_genes = []
        if row['gene_name'] in ppi_available_genes:
            ### get all connected genes
            connected_genes = ppi[ppi['Translated_protein_1'] == row['gene_name']]['Translated_protein_2'].tolist()
            connected_genes += ppi[ppi['Translated_protein_2'] == row['gene_name']]['Translated_protein_1'].tolist()
            connected_genes = list(set(connected_genes))
        else:
            connected_genes = []
        gene_df.loc[i, 'connected_genes'] = ', '.join(connected_genes)
        gene_df.loc[i, 'num_connected_genes'] = len(connected_genes)

    return gene_df

def number_of_unique_connected_genes(gene_df):
    unique_connected_genes = set()
    for connected_genes in gene_df['connected_genes']:
        if pd.notna(connected_genes):
            unique_connected_genes.update(connected_genes.split(', '))

    return list(unique_connected_genes), len(set(unique_connected_genes))

In [ ]:
def connect_direct_gene_to_drug(gene_df, drug_df = Drug_bank):
    ### return two df one explicity drug-gene each one or one with drug-gene conections grouped by gene, and one with drug-gene connections grouped by drug
    gene_drug_connections = []
    for i, gene in tqdm(enumerate(gene_df['gene_name'])):
        connected_drugs = drug_df[drug_df['gene'] == gene]['drug'].tolist()
        gene_drug_connections.append({
            'gene': gene,
            'connected_drugs': ', '.join(connected_drugs),
            'num_connected_drugs': len(connected_drugs)
        })
    gene_drug_df = pd.DataFrame(gene_drug_connections)  
    # Create explicit drug-gene connections DataFrame
    explicit_df = []
    for _, row in gene_df.iterrows():
        gene = row['gene_name']
        connected_drugs = drug_df[drug_df['gene'] == gene]
        if len(connected_drugs) > 0:
            for _, drug_row in connected_drugs.iterrows():
                explicit_df.append({
                    'gene': gene,
                    'drug': drug_row['drug'],
                    'polypeptide': drug_row['polypeptide'],
                    'gene_description': drug_row['gene_description'],
                    'action': drug_row['action'],
                    'specific_function': drug_row['specific_function'],
                    'toxicity': drug_row['toxicity'],
                    'categories': drug_row['categories']
                })
    
    explicit_df = pd.DataFrame(explicit_df)
    # Merge with original gene_df to include mutation information
    explicit_df = explicit_df.merge(gene_df[['gene_name', 'MUT_TYPE', 'q_value', 'empirical_q_value']], 
                                     left_on='gene', right_on='gene_name', how='left')
    
    return explicit_df, gene_drug_df

In [ ]:
def connect_indirect_gene_to_drug(indirect_gene_df, drug_df = Drug_bank):
    ### after running the connect_to_genes function, we can then connect the indirectly connected genes to drugs using the drug bank data
    ### return two df one with drug-indirect gene - risk gene explictly mentioned and one with drug-indirect gene - risk gene implicitly mentioned
    ### retrun andother df with grouped by drug with all the connected genes and risk genes mentioned
    # Extract all unique indirectly connected genes from the indirect_gene_df
    indirect_gene_df['gene_name'] = indirect_gene_df['gene_name'].str.upper()
    indirect_gene_df['connected_genes'] = indirect_gene_df['connected_genes'].str.upper()
    drug_df['gene'] = drug_df['gene'].str.upper()
    
    
    all_indirect_genes = []
    risk_gene_mapping = {}  # Map indirect gene to list of risk genes
    for idx, row in indirect_gene_df.iterrows():
        risk_gene = row['gene_name']
        connected_genes_str = row['connected_genes']
        if pd.notna(connected_genes_str) and connected_genes_str:
            connected_genes_list = [g.strip() for g in connected_genes_str.split(',')]
            all_indirect_genes.extend(connected_genes_list)
            # Map each indirect gene to its risk gene(s)
            for indirect_gene in connected_genes_list:
                if indirect_gene not in risk_gene_mapping:
                    risk_gene_mapping[indirect_gene] = []
                risk_gene_mapping[indirect_gene].append(risk_gene)

    # Get unique indirect genes
    unique_indirect_genes = list(set(all_indirect_genes))
    # Connect indirect genes to drugs
    drug_indirect_gene_risk_explicit = []
    for indirect_gene in tqdm(unique_indirect_genes):
        # Find drugs targeting this indirect gene
        connected_drugs = drug_df[drug_df['gene'] == indirect_gene]
        if len(connected_drugs) > 0:
            for _, drug_row in connected_drugs.iterrows():
                # Get all risk genes associated with this indirect gene
                
                associated_risk_genes = risk_gene_mapping.get(indirect_gene, [])
                for risk_gene in associated_risk_genes:
                    drug_indirect_gene_risk_explicit.append({
                        'drug': drug_row['drug'],
                        'indirect_gene': indirect_gene,
                        'risk_gene': risk_gene,
                        'polypeptide': drug_row['polypeptide'],
                        'gene_description': drug_row['gene_description'],
                        'action': drug_row['action'],
                        'specific_function': drug_row['specific_function'],
                        'toxicity': drug_row['toxicity'],
                        'categories': drug_row['categories']
                    })

    # Create DataFrame with explicit mapping
    df_explicit = pd.DataFrame(drug_indirect_gene_risk_explicit)

    # Create grouped DataFrame by drug
    if len(df_explicit) > 0:
        df_grouped = df_explicit.groupby('drug').agg({
            'indirect_gene': lambda x: ', '.join(sorted(set(x))),
            'risk_gene': lambda x: ', '.join(sorted(set(x))),
            'polypeptide': 'first',
            'action': 'first',
            'toxicity': 'first',
            'categories': 'first'
        }).reset_index()
        
        df_grouped.rename(columns={
            'indirect_gene': 'all_indirect_genes',
            'risk_gene': 'all_risk_genes'
        }, inplace=True)
    else:
        df_grouped = pd.DataFrame()

    return df_explicit, df_grouped

In [ ]:
def overlap_ehr_drugs(gene_drug_df, ehr_drug_df):
    ### return df with merged ehr data on gene data only keeping ehr drugs that overlap with gene_drug_df
    ehr_drug_df = ehr_drug_df.rename(columns={'feature': 'drug'})
    ### remove'_' from drug names in ehr_drug_df if there
    ehr_drug_df['drug'] = ehr_drug_df['drug'].str.replace('_', '')
    ### case match both gene_drug_df and ehr_drug_df drug names to upper case
    gene_drug_df['drug'] = gene_drug_df['drug'].str.upper()
    ehr_drug_df['drug'] = ehr_drug_df['drug'].str.upper()
    overlap_df = gene_drug_df.merge(ehr_drug_df, left_on='drug', right_on='drug', how='inner', suffixes=('_gene', '_ehr'))
    ### print number of drugs validated
    print(f"Number of overlapped medications {len(set(overlap_df['drug']))}")
    print(f"Out of {len(set(ehr_drug_df['drug']))} medications in EHR data")
    print(f"Overlapped medications are: {list(set(overlap_df['drug']))}")
    return overlap_df

In [ ]:
def connect_ehr_and_genetic_indirect(ehr_drug_df, genetic_gene_df):
    ### fix case sensitivity issue
    ehr_drug_df['feature'] = ehr_drug_df['feature'].str.upper()
    genetic_gene_df['drug'] = genetic_gene_df['drug'].str.upper()
    combined_results = []
    for i, row in ehr_drug_df.iterrows():
        drug_name = row['feature'].lstrip('_') # remove the _ prefix
        drug_name = drug_name.upper()  # Normalize to uppercase for matching
        # Check if this drug is connected to any genes in the genetic results
        for j, gene_row in genetic_gene_df.iterrows():
            if drug_name == gene_row['drug']:
                combined_results.append({
                    'drug': drug_name,
                    'gene_drug': gene_row['drug'],
                    'indirect_gene' : gene_row['indirect_gene'],
                    'risk_gene' : gene_row['risk_gene'],
                    'polypeptide': gene_row['polypeptide'],
                    'gene_description': gene_row['gene_description'],
                    'action': gene_row['action'],
                    'specific_function': gene_row['specific_function'],
                    'toxicity': gene_row['toxicity'],
                    'categories': gene_row['categories'],
                    'xgb_importance': row['xgb_importance'],
                    'xgb_absolute_importance': row['xgb_absolute_Importance'],
                    'rf_importance': row['rf_importance'],
                    'univariate_hr': row['univariate_hr'],
                    'multivariate_hr': row['multivariate_hr']
                    # 'genetic_additional_info': gene_row.to_dict()
                })
    overlap_df = pd.DataFrame(combined_results)
    print(f"Number of overlapped medications {len(set(overlap_df['drug']))}")
    print(f"Out of {len(set(ehr_drug_df['feature']))} medications in EHR data")
    print(f"Overlapped medications are: {list(set(overlap_df['drug']))}")
    return overlap_df

In [ ]:
import plotly.graph_objects as go

def plot_sankey_drug_to_neighbor_to_riskgene(indirect_explicit_df, drug_name):
    """
    Plots a Sankey diagram for a given drug showing connections:
    drug → indirect_gene (neighbor) → risk_gene
    Ensures each node appears only once. Colors: blue (drug), green (neighbor), red (risk gene).
    """
    # Filter for the selected drug (case-insensitive)
    df = indirect_explicit_df[indirect_explicit_df['drug'].str.upper() == drug_name.upper()]
    if df.empty:
        print(f"No data for drug: {drug_name}")
        return

    # Unique node labels
    drug_label = drug_name.upper()
    indirect_genes = sorted(df['indirect_gene'].unique())
    risk_genes = sorted(df['risk_gene'].unique())

    # Build node list: drug, indirect genes, risk genes
    node_labels = [drug_label] + indirect_genes + risk_genes
    node_indices = {label: i for i, label in enumerate(node_labels)}

    # Links: drug → indirect_gene
    links_1 = df.groupby(['drug', 'indirect_gene']).size().reset_index(name='count')
    # Links: indirect_gene → risk_gene
    links_2 = df.groupby(['indirect_gene', 'risk_gene']).size().reset_index(name='count')

    # Sankey link lists
    sources = []
    targets = []
    values = []
    # drug → indirect_gene
    for _, row in links_1.iterrows():
        sources.append(node_indices[drug_label])
        targets.append(node_indices[row['indirect_gene']])
        values.append(row['count'])
    # indirect_gene → risk_gene
    for _, row in links_2.iterrows():
        sources.append(node_indices[row['indirect_gene']])
        targets.append(node_indices[row['risk_gene']])
        values.append(row['count'])

    # Colors: blue (drug), green (neighbor), red (risk gene)
    node_colors = ["#2196f3"] + ["#43a047"]*len(indirect_genes) + ["#e53935"]*len(risk_genes)

    # Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values
        )
    )])
    fig.update_layout(title_text=f"Sankey Diagram for {drug_label} (Drug → Neighbor → Risk Gene)", font_size=12)
    fig.show()

# Example usage:
# plot_sankey_drug_to_neighbor_to_riskgene(indirect_explicit, 'UREA')


In [ ]:
from plotly.subplots import make_subplots

def plot_multiple_sankeys(indirect_explicit_df, drug_list, cols=2, title_prefix="", height=2200, width=2000, gene_targets=None, family_to_genes=None):
    """
    Plots multiple Sankey diagrams for a list of drugs in a grid layout (default 2 columns).
    Each subplot: drug → neighbor → risk gene (blue-green-red).
    Uniform node/link style for publication-quality consistency.
    """
    import plotly.graph_objects as go
    import math
    rows = math.ceil(len(drug_list) / cols)
    subplot_titles = [drug.title() for drug in drug_list]
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=subplot_titles,
        specs=[[{'type': 'sankey'}] * cols for _ in range(rows)],
        vertical_spacing=0.05,
        horizontal_spacing=0.03
    )

    for idx, drug in enumerate(drug_list):
        row = (idx // cols) + 1
        col = (idx % cols) + 1
        df = indirect_explicit_df[indirect_explicit_df['drug'].str.upper() == drug.upper()]
        if df.empty:
            continue
        drug_label = drug.upper()
        indirect_genes = sorted(df['indirect_gene'].unique())
        risk_genes = sorted(df['risk_gene'].unique())
        node_labels = [drug_label] + indirect_genes + risk_genes
        node_indices = {label: i for i, label in enumerate(node_labels)}
        # Links: drug → indirect_gene
        links_1 = df.groupby(['drug', 'indirect_gene']).size().reset_index(name='count')
        # Links: indirect_gene → risk_gene
        links_2 = df.groupby(['indirect_gene', 'risk_gene']).size().reset_index(name='count')
        sources = []
        targets = []
        values = []
        for _, row_link in links_1.iterrows():
            sources.append(node_indices[drug_label])
            targets.append(node_indices[row_link['indirect_gene']])
            values.append(1)  # Uniform thickness
        for _, row_link in links_2.iterrows():
            sources.append(node_indices[row_link['indirect_gene']])
            targets.append(node_indices[row_link['risk_gene']])
            values.append(1)  # Uniform thickness
        node_colors = []
        for label in node_labels:
            if label == drug_label:
                node_colors.append('rgba(31, 119, 180, 0.9)')  # Blue
            elif gene_targets and label in gene_targets:
                node_colors.append('rgba(44, 160, 44, 0.9)')   # Green
            elif label in indirect_genes:
                node_colors.append('rgba(44, 160, 44, 0.9)')   # Green
            else:
                node_colors.append('rgba(214, 39, 40, 0.9)')   # Red
        fig.add_trace(
            go.Sankey(
                arrangement='snap',
                valueformat='.0f',
                node=dict(
                    pad=8,
                    thickness=5,
                    line=dict(color="black", width=0.5),
                    label=node_labels,
                    color=node_colors
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    color='rgba(0, 0, 0, 0.2)'
                ),
                textfont=dict(size=15, color='black', family='Arial')
            ),
            row=row,
            col=col
        )
    fig.update_layout(
        title={
            'text': f"{title_prefix}Individual Drug Pathways<br>",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 22}
        },
        height=height,
        width=width,
        showlegend=False,
        font=dict(size=14),
        margin=dict(t=100, b=20, l=20, r=20)
    )
    # Update subplot titles (drug names)
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=18, color='black', family='Arial')
    # Add gene family legend if families are present
    if family_to_genes:
        all_gene_families = sorted(family_to_genes.keys())
        if all_gene_families:
            legend_lines = ["<b>Gene Families:</b>"]
            for family in all_gene_families:
                genes = family_to_genes[family]
                if len(genes) <= 3:
                    gene_list = ", ".join(genes)
                else:
                    gene_list = ", ".join(genes[:3]) + f", ... (+{len(genes)-3})"
                legend_lines.append(f"• <b>{family}</b>: {gene_list}")
            legend_text = "<br>".join(legend_lines)
            fig.add_annotation(
                text=legend_text,
                xref="paper", yref="paper",
                x=0.01, y=1,
                xanchor="left", yanchor="top",
                showarrow=False,
                font=dict(size=9, family="Arial"),
                align="left",
                bgcolor="rgba(255, 255, 255, 0.9)",
                bordercolor="black",
                borderwidth=1,
                borderpad=6
            )
    fig.show()

# Example usage:
# plot_multiple_sankeys(indirect_explicit, ['UREA', 'ARTENIMOL', ...], gene_targets=set(...), family_to_genes={...})


In [ ]:
def group_ehr_overlap_by_drug(hpv_pos_indirect_ehr_overlap):
    """
    Group `hpv_pos_indirect_ehr_overlap` by `drug`.

    - For `indirect_gene` and `risk_gene`: collect all values across rows,
      split comma/semicolon/pipe-separated entries, strip, deduplicate, sort, and join with ', '.
    - For importance/HR columns keep the first non-null value per drug:
      `xgb_importance`, `rf_importance`, `univariate_hr`, `multivariate_hr`.

    Returns a DataFrame with columns:
        ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    """
    import re
    import pandas as pd
    # Defensive: ensure expected columns exist
    expected = ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    hpv_pos_indirect_ehr_overlap = hpv_pos_indirect_ehr_overlap[expected]

    # Group and aggregate
    grouped = hpv_pos_indirect_ehr_overlap.groupby('drug').agg({
        'indirect_gene': lambda s: ','.join(sorted(set(s.dropna().astype(str)))),
        'risk_gene': lambda s: ','.join(sorted(set(s.dropna().astype(str)))),
        'xgb_importance': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'rf_importance': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'univariate_hr': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'multivariate_hr': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
    }).reset_index()

    out_cols = ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    result = grouped[out_cols]
    return result


### HPV+

#### Genes

In [ ]:
hpv_positive_genes[hpv_positive_genes['gene_name'] == 'RFC4']

In [ ]:
hpv_pos_som_genes = hpv_positive_som_genes['Gene'].unique()
print(f"Number of unique genes in HPV positive somatic mutation top genes: {len(hpv_pos_som_genes)}")
drugbank_genes = list(set(Drug_bank['gene'].dropna().values))
overlap_genes = set(hpv_pos_som_genes) & set(drugbank_genes)
print(f"Number of overlapping genes between HPV positive somatic mutation top genes and DrugBank genes: {len(overlap_genes)}")
print(f"Overlapping genes: {overlap_genes}")

## immediate neighbors of these genes in the PPI network
hpv_pos_som_genes = hpv_positive_som_genes['Gene'].unique()
ppi_genes = list(set(protein_interaction['Translated_protein_1'].dropna().values) | set(protein_interaction['Translated_protein_2'].dropna().values))
overlap_genes = set(hpv_pos_som_genes) & set(ppi_genes)
#print(f"Number of overlapping genes between HPV positive somatic mutation top genes and PPI genes: {len(overlap_genes)}")
# print(f"Overlapping genes: {overlap_genes}")

## number of immediate neighbors of the top genes in the PPI network
usable_protein_interaction = protein_interaction[protein_interaction['combined_score'] > 700]
protein_interaction_neighbors = usable_protein_interaction[usable_protein_interaction['Translated_protein_1'].isin(overlap_genes)]
### remove duplicates and hpv pos som genes from the neighbors
immediate_neighbors = protein_interaction_neighbors['Translated_protein_2'].unique()
print(f"Number of immediate neighbors of the top genes in the PPI network: {len(immediate_neighbors)}")

### number of immediate neighbors or key risk genes that are in drugbank
immediate_neighbors_and_risk = list(list(immediate_neighbors) + list(hpv_pos_som_genes))
immediate_neighbors_and_risk_in_drugbank = set(immediate_neighbors_and_risk) & set(drugbank_genes)
print(f"Number of immediate neighbors or key risk genes that are in DrugBank: {len(immediate_neighbors_and_risk_in_drugbank)}")
print(f"Immediate neighbors or key risk genes that are in DrugBank: {immediate_neighbors_and_risk_in_drugbank}")

In [ ]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()

hpv_positive_combined_genes


In [ ]:
hpv_positive_combined_genes[hpv_positive_combined_genes['gene_name'] == 'SOX2']

#### Expand through PPI

In [ ]:
hpv_positive_connected_genes = connect_to_genes(hpv_positive_combined_genes)

In [ ]:
hpv_positive_connected_genes.sort_values('num_connected_genes', ascending=False).head(20)

In [ ]:
number_of_unique_connected_genes(hpv_positive_connected_genes)

#### Connect to drugs

In [ ]:
hpv_pos_direct_gene_drug_connections_explicit, hpv_pos_direct_gene_drug_connections_grouped = connect_direct_gene_to_drug(hpv_positive_connected_genes)

In [ ]:
hpv_pos_direct_gene_drug_connections_explicit

In [ ]:
print(f"Number of directly connected drugs: {len(set(hpv_pos_direct_gene_drug_connections_explicit['drug']))}")

In [ ]:
hpv_pos_df_indirect_explicit, hpv_pos_df_indirect_grouped = connect_indirect_gene_to_drug(hpv_positive_connected_genes)

In [ ]:
print(f"Number of unique genes that are targetable: {len(set(hpv_pos_df_indirect_explicit['indirect_gene']))}")
print(f"Number of unique risk genes connected to drugs: {len(set(hpv_pos_direct_gene_drug_connections_explicit['gene']))}")
print(f"Number of unique risk genes or indirectly connected genes connected to drugs: {len(set(hpv_pos_df_indirect_explicit['risk_gene']).union(set(hpv_pos_df_indirect_explicit['indirect_gene'])).union(set(hpv_pos_direct_gene_drug_connections_explicit['gene'])))}")

In [ ]:
hpv_pos_df_indirect_explicit

In [ ]:
print(f'Number of medications identified {len(set(hpv_pos_df_indirect_explicit['drug']))}')

In [ ]:
print(f'Total number of medications identified {len(set(hpv_pos_df_indirect_explicit['drug']).union(set(hpv_pos_direct_gene_drug_connections_explicit['drug'])))}')

#### EHR match

In [ ]:
hpv_pos_direct_ehr_overlap = overlap_ehr_drugs(hpv_pos_direct_gene_drug_connections_explicit, ehr_hpv_pos_drugs)

In [ ]:
hpv_pos_indirect_ehr_overlap = connect_ehr_and_genetic_indirect(ehr_hpv_pos_drugs, hpv_pos_df_indirect_explicit)

In [ ]:
grouped_ehr_pos_results = group_ehr_overlap_by_drug(hpv_pos_indirect_ehr_overlap)

In [ ]:
grouped_ehr_pos_results

In [ ]:
grouped_ehr_pos_results.to_csv('Results/Final results/hpv_positive_ehr_genetic_overlap_grouped.csv', index=False)

In [ ]:
hpv_pos_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)[['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']]

In [ ]:
ehr_risk_genes = hpv_pos_indirect_ehr_overlap['risk_gene'].unique()
hpv_positive_combined_genes[hpv_positive_combined_genes['gene_name'].isin(ehr_risk_genes)]

In [ ]:
hpv_pos_indirect_ehr_overlap['drug'].nunique()

In [ ]:
hpv_pos_indirect_ehr_overlap['risk_gene'].unique()

In [ ]:
hpv_pos_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)['drug'].unique()[:10]

In [ ]:
hpv_pos_indirect_ehr_overlap.to_csv('Results/Integrated results/hpv_positive_indirect_ehr_overlap.csv', index=False)

In [ ]:
# top_10_pos_drugs = list(hpv_pos_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)['drug'].unique()[:10])
# hpv_pos_indirect_ehr_overlap[hpv_pos_indirect_ehr_overlap['drug'].isin(top_10_pos_drugs)]

In [ ]:
hpv_pos_indirect_ehr_overlap

In [ ]:
# drugs_interested = ['ACETAMINOPHEN', 'HEPARIN', 'LEVOTHYROXINE', 'DEXAMETHASONE', 'ATORVASTATIN', 'LOSARTAN']
plot_multiple_sankeys(hpv_pos_df_indirect_explicit, list(grouped_ehr_pos_results.sort_values(by='xgb_importance')['drug'].unique()[:10]), title_prefix="HPV Positive - ")

In [ ]:
drugs_interested = ['LEVOTHYROXINE', 'ENOXAPARIN', 'BACITRACIN', 'ONDANSETRON', 'HEPARIN', 'OXYMETAZOLINE', 'DEXAMETHASONE', 'ACETAMINOPHEN', 'MORPHINE', 'ATORVASTATIN']
plot_multiple_sankeys(hpv_pos_df_indirect_explicit, drugs_interested, title_prefix="HPV Positive - ")

In [ ]:
### not showing glucose as it is a common medication that may be used in various contexts and may not be specific to the disease context we are studying, 
# and it may also have a high importance score due to its widespread use rather than a specific biological connection to the disease. 
# Therefore, we can choose to exclude it from the visualization to focus on more relevant drug-gene interactions.
### it has high connections to genes but is not specific to the disease context and may not provide meaningful insights in the analysis, 
# so we can choose to exclude it from the visualization to focus on more relevant drug-gene interactions.
drugs_interested = ['ACETAMINOPHEN', 'HEPARIN', 'LEVOTHYROXINE', 'DEXAMETHASONE', 'ATORVASTATIN', 'LOSARTAN']
plot_multiple_sankeys(hpv_pos_df_indirect_explicit, drugs_interested, title_prefix="HPV Positive - ")

In [ ]:
plot_sankey_drug_to_neighbor_to_riskgene(hpv_pos_df_indirect_explicit, 'MORPHINE')

In [ ]:
overlap_ehr_drugs(hpv_pos_direct_gene_drug_connections_explicit, ehr_hpv_pos_drugs)

### HPV-

In [ ]:
hpv_negative_genes

In [ ]:
hpv_neg_som_genes = hpv_negative_som_genes['Gene'].unique()
print(f"Number of unique genes in HPV negative somatic mutation top genes: {len(hpv_neg_som_genes)}")
drugbank_genes = list(set(Drug_bank['gene'].dropna().values))
overlap_genes = set(hpv_neg_som_genes) & set(drugbank_genes)
print(f"Number of overlapping genes between HPV negative somatic mutation top genes and DrugBank genes: {len(overlap_genes)}")
print(f"Overlapping genes: {overlap_genes}")

## immediate neighbors of these genes in the PPI network
hpv_neg_som_genes = hpv_negative_som_genes['Gene'].unique()
ppi_genes = list(set(protein_interaction['Translated_protein_1'].dropna().values) | set(protein_interaction['Translated_protein_2'].dropna().values))
overlap_genes = set(hpv_neg_som_genes) & set(ppi_genes)
#print(f"Number of overlapping genes between HPV negative somatic mutation top genes and PPI genes: {len(overlap_genes)}")
# print(f"Overlapping genes: {overlap_genes}")

## number of immediate neighbors of the top genes in the PPI network
usable_protein_interaction = protein_interaction[protein_interaction['combined_score'] > 700]
protein_interaction_neighbors = usable_protein_interaction[usable_protein_interaction['Translated_protein_1'].isin(overlap_genes)]
### remove duplicates and hpv neg som genes from the neighbors
immediate_neighbors = protein_interaction_neighbors['Translated_protein_2'].unique()
print(f"Number of immediate neighbors of the top genes in the PPI network: {len(immediate_neighbors)}")

### number of immediate neighbors or key risk genes that are in drugbank
immediate_neighbors_and_risk = list(list(immediate_neighbors) + list(hpv_neg_som_genes))
immediate_neighbors_and_risk_in_drugbank = set(immediate_neighbors_and_risk) & set(drugbank_genes)
print(f"Number of immediate neighbors or key risk genes that are in DrugBank: {len(immediate_neighbors_and_risk_in_drugbank)}")
print(f"Immediate neighbors or key risk genes that are in DrugBank: {immediate_neighbors_and_risk_in_drugbank}")

In [ ]:
### combine hpv positive somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()

hpv_negative_combined_genes

In [ ]:
hpv_negative_combined_genes[hpv_negative_combined_genes['gene_name'] == 'BCL6']

In [ ]:
hpv_negative_som_genes[hpv_negative_som_genes['Gene'] == 'TP53']

#### Expand through PPI

In [ ]:
hpv_negative_connected_genes = connect_to_genes(hpv_negative_combined_genes)

In [ ]:
number_of_unique_connected_genes(hpv_negative_connected_genes)

#### Connect to drugs

In [ ]:
hpv_neg_direct_gene_drug_connections_explicit, hpv_neg_direct_gene_drug_connections_grouped = connect_direct_gene_to_drug(hpv_negative_connected_genes)

In [ ]:
hpv_neg_direct_gene_drug_connections_explicit

In [ ]:
hpv_neg_direct_gene_drug_connections_explicit[hpv_neg_direct_gene_drug_connections_explicit['drug']=='Acetylsalicylic acid']

In [ ]:
hpv_neg_df_indirect_explicit, hpv_neg_df_indirect_grouped = connect_indirect_gene_to_drug(hpv_negative_connected_genes)

In [ ]:
hpv_neg_df_indirect_grouped

In [ ]:
hpv_neg_df_indirect_grouped[hpv_neg_df_indirect_grouped['drug']=='Dasatinib']

#### EHR match

In [ ]:
hpv_neg_direct_ehr_overlap = overlap_ehr_drugs(hpv_neg_direct_gene_drug_connections_explicit, ehr_hpv_neg_drugs)

In [ ]:
hpv_neg_direct_ehr_overlap

In [ ]:
hpv_neg_indirect_ehr_overlap = overlap_ehr_drugs(hpv_neg_df_indirect_explicit, ehr_hpv_neg_drugs)

In [ ]:
hpv_neg_indirect_ehr_overlap = connect_ehr_and_genetic_indirect(ehr_hpv_neg_drugs, hpv_neg_df_indirect_explicit)

In [ ]:
hpv_neg_indirect_ehr_overlap

In [ ]:
top_10_neg_drugs = list(hpv_neg_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)['drug'].unique()[:10])
hpv_neg_indirect_ehr_overlap_top_10_drugs = hpv_neg_indirect_ehr_overlap[hpv_neg_indirect_ehr_overlap['drug'].isin(top_10_neg_drugs)]
ehr_risk_genes = hpv_neg_indirect_ehr_overlap_top_10_drugs['risk_gene'].unique()
hpv_negative_combined_genes[hpv_negative_combined_genes['gene_name'].isin(ehr_risk_genes)]

In [ ]:
ehr_risk_genes.sort()
ehr_risk_genes

In [ ]:
hpv_neg_indirect_ehr_overlap.to_csv('Results/Integrated results/hpv_negative_indirect_ehr_overlap.csv', index=False)

In [ ]:
grouped_ehr_neg_results = group_ehr_overlap_by_drug(hpv_neg_indirect_ehr_overlap)
### change "acetylsalicylic acid" to "aspirin"
grouped_ehr_neg_results['drug'] = grouped_ehr_neg_results['drug'].str.replace('ACETYLSALICYLIC ACID', 'ASPIRIN')

In [ ]:
grouped_ehr_neg_results

In [ ]:
grouped_ehr_neg_results[grouped_ehr_neg_results['drug'] == 'MELATONIN']['risk_gene'].to_list()

In [ ]:
grouped_ehr_neg_results[grouped_ehr_neg_results['drug'] == 'TRAMADOL']['indirect_gene'].to_list()

In [ ]:
list(grouped_ehr_neg_results['drug'].str.lower())

In [ ]:
top_10_neg_drugs = list(hpv_neg_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)['drug'].unique()[:10])
hpv_neg_indirect_ehr_overlap[hpv_neg_indirect_ehr_overlap['drug'].isin(top_10_neg_drugs)]

In [ ]:
grouped_ehr_neg_results.to_csv('Results/Integrated results/hpv_negative_grouped_ehr_overlap.csv', index=False)

In [ ]:
interested_drugs_neg = ['IPRATROPIUM', 'THIAMINE', 'MELATONIN', 'PREDNISOLONE', 'TRAMADOL', 'FUROSEMIDE', 'ATORVASTATIN', 'HYDRALAZINE', 'ALBUTEROL', 'POTASSIUM CHLORIDE']
plot_multiple_sankeys(hpv_neg_df_indirect_explicit, interested_drugs_neg, title_prefix="HPV Negative - ")

In [ ]:
interested_drugs_neg = ['METHYLPREDNISOLONE', 'MELATONIN', 'LEVOTHYROXINE', 'ACETAMINOPHEN', 'ATORVASTATIN', 'THIAMINE']
plot_multiple_sankeys(hpv_neg_df_indirect_explicit, interested_drugs_neg, title_prefix="HPV Negative - ")

In [ ]:
plot_sankey_drug_to_neighbor_to_riskgene(hpv_neg_indirect_ehr_overlap, 'aspirin')

In [ ]:
def group_ehr_overlap_by_drug(hpv_pos_indirect_ehr_overlap):
    """
    Group `hpv_pos_indirect_ehr_overlap` by `drug`.

    - For `indirect_gene` and `risk_gene`: collect all values across rows,
      split comma/semicolon/pipe-separated entries, strip, deduplicate, sort, and join with ', '.
    - For importance/HR columns keep the first non-null value per drug:
      `xgb_importance`, `rf_importance`, `univariate_hr`, `multivariate_hr`.

    Returns a DataFrame with columns:
        ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    """
    import re
    import pandas as pd

    # Helper to flatten comma/semicolon/pipe separated strings in a Series
    def _flatten_and_join(series):
        parts = []
        for val in series.dropna().astype(str):
            # split on comma, semicolon or pipe
            for piece in re.split(r"[,;|]", val):
                p = piece.strip()
                if p:
                    parts.append(p)
        unique_sorted = sorted(set(parts))
        return ', '.join(unique_sorted)

    # Defensive: ensure expected columns exist
    expected = ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    missing = [c for c in expected if c not in hpv_pos_indirect_ehr_overlap.columns]
    if missing:
        raise ValueError(f"Input DataFrame is missing required columns: {missing}")

    # Group and aggregate
    grouped = hpv_pos_indirect_ehr_overlap.groupby('drug').agg({
        # Keep `indirect_gene` joined as-is (no flattening/splitting)
        'indirect_gene': lambda s: ','.join(s.dropna().astype(str)),
        # For risk genes, still flatten/deduplicate/sort and join
        'risk_gene': lambda s: _flatten_and_join(s),
        'xgb_importance': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'rf_importance': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'univariate_hr': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'multivariate_hr': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
    }).reset_index()

    out_cols = ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    result = grouped[out_cols]
    return result
